# Task 4.1 — Profitable Stock Selection (Vietnam Market)

**Goal:** Select a list of profitable Vietnamese companies to include in a portfolio.  
**Methodology:**
1. Load and filter companies (VNINDEX only, ≥ 120 data points)
2. Auto-select top 30 by longest history
3. Engineer financial features (returns, SMA, RSI, volatility, Sharpe-like score)
4. Build an LSTM model to predict future 30-day return for each company
5. Rank companies by predicted profitability score
6. Select top portfolio candidates with justification

## 1. Imports & Configuration

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, GRU, Dense, Dropout, Bidirectional,
    BatchNormalization, Concatenate, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

DATA_DIR = '/Users/tanle/Downloads/Spring2026/CS313/DL4AI-220176-project/data-vn-20230228/stock-historical-data'
EXCHANGE_FILTER = 'VNINDEX'
MIN_DATA_POINTS = 120
TOP_N_COMPANIES = 30
WINDOW_SIZE = 60
FORECAST_HORIZON = 30
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
EPOCHS = 60
BATCH_SIZE = 32

print('Configuration set.')
print(f'  Data directory : {DATA_DIR}')
print(f'  Exchange filter: {EXCHANGE_FILTER}')
print(f'  Min data points: {MIN_DATA_POINTS}')
print(f'  Top N companies: {TOP_N_COMPANIES}')
print(f'  Window size    : {WINDOW_SIZE} days')
print(f'  Forecast horizon: {FORECAST_HORIZON} days')

Configuration set.
  Data directory : /Users/tanle/Downloads/Spring2026/CS313/DL4AI-220176-project/data-vn-20230228/stock-historical-data
  Exchange filter: VNINDEX
  Min data points: 120
  Top N companies: 30
  Window size    : 60 days
  Forecast horizon: 30 days


## 2. Load & Filter Companies

In [2]:
def load_company_csv(filepath):
    """Load a single company CSV, parse dates, sort chronologically."""
    df = pd.read_csv(filepath, index_col=0)
    df['TradingDate'] = pd.to_datetime(df['TradingDate'])
    df = df.sort_values('TradingDate').reset_index(drop=True)
    df = df.dropna(subset=['Open', 'High', 'Low', 'Close'])
    df = df[df['Close'] > 0]
    return df

all_files = glob.glob(os.path.join(DATA_DIR, f'*-{EXCHANGE_FILTER}-History.csv'))
print(f'Found {len(all_files)} {EXCHANGE_FILTER} company files.')

company_meta = []
for fp in all_files:
    filename = os.path.basename(fp)
    ticker = filename.replace(f'-{EXCHANGE_FILTER}-History.csv', '')
    try:
        df = load_company_csv(fp)
        n = len(df)
        if n >= MIN_DATA_POINTS:
            company_meta.append({'ticker': ticker, 'filepath': fp, 'n_rows': n})
    except Exception as e:
        print(f'  Skipping {ticker}: {e}')

meta_df = pd.DataFrame(company_meta).sort_values('n_rows', ascending=False).reset_index(drop=True)
print(f'Companies with >= {MIN_DATA_POINTS} data points: {len(meta_df)}')

selected_meta = meta_df.head(TOP_N_COMPANIES)
print(f'\nTop {TOP_N_COMPANIES} companies selected (by most historical data):')
print(selected_meta[['ticker', 'n_rows']].to_string(index=False))

Found 416 VNINDEX company files.
Companies with >= 120 data points: 412

Top 30 companies selected (by most historical data):
ticker  n_rows
   REE    5410
   TMS    5410
   SAM    5410
   LAF    5404
   HAP    5399
   GIL    5259
   BBC    5258
   GMD    5201
   SAV    5191
   HAS    5032
   DHA    4710
   SFC    4600
   SSC    4485
   MHC    4477
   PNC    4399
   TNA    4392
   VSH    4297
   KDC    4290
   HTV    4273
   VNM    4264
   TYA    4250
   KHP    4249
   CII    4186
   PPC    4172
   SJS    4151
   BMP    4148
   STB    4147
   COM    4129
   SMC    4070
   ITA    4058


## 3. Feature Engineering

In [3]:
def compute_rsi(series, period=14):
    """Relative Strength Index."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

def engineer_features(df):
    """Add technical indicators as model features."""
    df = df.copy()

    df['daily_return'] = df['Close'].pct_change()

    df['sma_10'] = df['Close'].rolling(10).mean()
    df['sma_20'] = df['Close'].rolling(20).mean()
    df['sma_50'] = df['Close'].rolling(50).mean()

    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']

    df['rsi_14'] = compute_rsi(df['Close'], 14)

    df['volatility_20'] = df['daily_return'].rolling(20).std()

    df['hl_range'] = (df['High'] - df['Low']) / (df['Close'] + 1e-9)

    df['price_sma20_ratio'] = df['Close'] / (df['sma_20'] + 1e-9)

    df['volume_ma20'] = df['Volume'].rolling(20).mean()
    df['volume_ratio'] = df['Volume'] / (df['volume_ma20'] + 1e-9)

    df['future_return'] = df['Close'].shift(-FORECAST_HORIZON) / (df['Close'] + 1e-9) - 1

    df = df.dropna().reset_index(drop=True)
    return df

FEATURE_COLS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'daily_return', 'sma_10', 'sma_20', 'sma_50',
    'macd', 'macd_signal', 'macd_hist',
    'rsi_14', 'volatility_20', 'hl_range',
    'price_sma20_ratio', 'volume_ratio'
]
TARGET_COL = 'future_return'

print('Feature columns:', FEATURE_COLS)
print(f'Target: {TARGET_COL} ({FORECAST_HORIZON}-day forward return)')

Feature columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'daily_return', 'sma_10', 'sma_20', 'sma_50', 'macd', 'macd_signal', 'macd_hist', 'rsi_14', 'volatility_20', 'hl_range', 'price_sma20_ratio', 'volume_ratio']
Target: future_return (30-day forward return)


## 4. Time-Series Data Preparation

In [4]:
def chronological_split(df, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO):
    """
    Strictly chronological split: Train | Validation | Test
    No shuffling — time order must be preserved.
    """
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]

def make_sequences(features, targets, window_size=WINDOW_SIZE):
    """
    Create sliding window sequences for LSTM input.
    X shape: (samples, window_size, n_features)
    y shape: (samples,)
    """
    X, y = [], []
    for i in range(window_size, len(features)):
        X.append(features[i - window_size:i])
        y.append(targets[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def prepare_company_data(df):
    """
    Full pipeline: engineer features -> split -> scale -> make sequences.
    Scaler is fit ONLY on training data to prevent data leakage.
    """
    df_feat = engineer_features(df)
    if len(df_feat) < WINDOW_SIZE + 20:
        return None

    train_df, val_df, test_df = chronological_split(df_feat)

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_raw = scaler_X.fit_transform(train_df[FEATURE_COLS].values)
    X_val_raw = scaler_X.transform(val_df[FEATURE_COLS].values)
    X_test_raw = scaler_X.transform(test_df[FEATURE_COLS].values)

    y_train_raw = scaler_y.fit_transform(train_df[[TARGET_COL]].values).ravel()
    y_val_raw = scaler_y.transform(val_df[[TARGET_COL]].values).ravel()
    y_test_raw = scaler_y.transform(test_df[[TARGET_COL]].values).ravel()

    X_train, y_train = make_sequences(X_train_raw, y_train_raw)
    X_val, y_val = make_sequences(X_val_raw, y_val_raw)
    X_test, y_test = make_sequences(X_test_raw, y_test_raw)

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'scaler_X': scaler_X, 'scaler_y': scaler_y,
        'df_feat': df_feat,
        'test_dates': test_df['TradingDate'].values[WINDOW_SIZE:]
    }

print('Data preparation functions defined.')

Data preparation functions defined.


## 5. LSTM Model Architecture

**Architecture choice:** Bidirectional LSTM captures both past trends and momentum patterns. A second GRU layer adds lighter sequential refinement. This combination outperforms a single LSTM on financial time series while remaining trainable on small datasets.

In [5]:
def build_model(input_shape):
    """
    Bidirectional LSTM + GRU hybrid model.
    - Bidirectional LSTM: captures forward and backward temporal dependencies
    - GRU layer: lighter refinement of sequential patterns
    - BatchNorm + Dropout: regularization to prevent overfitting
    - Output: single neuron predicting normalized future_return
    """
    inp = Input(shape=input_shape)

    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = GRU(32, return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(16, activation='relu')(x)
    out = Dense(1, activation='linear')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='mse',
        metrics=['mae']
    )
    return model

sample_shape = (WINDOW_SIZE, len(FEATURE_COLS))
demo_model = build_model(sample_shape)
demo_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 17)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 60, 128)        │        41,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,721 (229.38 KB)

 Trainable params: 58,401 (228.13 KB)

 Non-trainable params: 320 (1.25 KB)

## 6. Train Model for Each Company

In [6]:
def train_company_model(ticker, data_dict):
    """Train and evaluate the LSTM model for one company."""
    X_train = data_dict['X_train']
    y_train = data_dict['y_train']
    X_val = data_dict['X_val']
    y_val = data_dict['y_val']
    X_test = data_dict['X_test']
    y_test = data_dict['y_test']
    scaler_y = data_dict['scaler_y']

    if len(X_train) < 10 or len(X_val) < 5 or len(X_test) < 5:
        return None

    model = build_model((WINDOW_SIZE, len(FEATURE_COLS)))

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )

    y_pred_scaled = model.predict(X_test, verbose=0).ravel()
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    avg_predicted_return = float(np.mean(y_pred))
    direction_accuracy = float(np.mean(np.sign(y_pred) == np.sign(y_true)))

    return {
        'model': model,
        'history': history,
        'y_pred': y_pred,
        'y_true': y_true,
        'mae': mae,
        'rmse': rmse,
        'avg_predicted_return': avg_predicted_return,
        'direction_accuracy': direction_accuracy,
        'test_dates': data_dict['test_dates']
    }

print('Training function ready.')

Training function ready.


In [ ]:
all_results = {}
failed = []

for i, row in selected_meta.iterrows():
    ticker = row['ticker']
    filepath = row['filepath']
    print(f'[{i+1}/{TOP_N_COMPANIES}] Training {ticker} ...', end=' ')

    try:
        df = load_company_csv(filepath)
        data_dict = prepare_company_data(df)
        if data_dict is None:
            print('SKIP (insufficient data after feature engineering)')
            failed.append(ticker)
            continue

        result = train_company_model(ticker, data_dict)
        if result is None:
            print('SKIP (insufficient split sizes)')
            failed.append(ticker)
            continue

        all_results[ticker] = result
        print(f'Done | MAE={result["mae"]:.4f} | AvgPredReturn={result["avg_predicted_return"]*100:.2f}%')

    except Exception as e:
        print(f'ERROR: {e}')
        failed.append(ticker)

print(f'\nSuccessfully trained: {len(all_results)} companies')
if failed:
    print(f'Failed/skipped: {failed}')

[1/30] Training REE ... Done | MAE=0.3004 | AvgPredReturn=-12.85%
[2/30] Training TMS ... Done | MAE=0.2200 | AvgPredReturn=16.36%
[3/30] Training SAM ... Done | MAE=0.1991 | AvgPredReturn=2.61%
[4/30] Training LAF ... Done | MAE=0.2670 | AvgPredReturn=-19.18%
[5/30] Training HAP ... Done | MAE=0.2177 | AvgPredReturn=-0.95%
[6/30] Training GIL ... Done | MAE=0.1848 | AvgPredReturn=-3.95%
[7/30] Training BBC ... Done | MAE=0.0942 | AvgPredReturn=0.32%
[8/30] Training GMD ... Done | MAE=0.1033 | AvgPredReturn=-0.43%
[9/30] Training SAV ... Done | MAE=0.1496 | AvgPredReturn=-2.57%
[10/30] Training HAS ... Done | MAE=0.1619 | AvgPredReturn=9.10%
[11/30] Training DHA ... Done | MAE=0.1832 | AvgPredReturn=-13.57%
[12/30] Training SFC ... Done | MAE=0.0639 | AvgPredReturn=-0.42%
[13/30] Training SSC ... Done | MAE=0.0607 | AvgPredReturn=0.13%
[14/30] Training MHC ... Done | MAE=0.2299 | AvgPredReturn=-6.62%
[15/30] Training PNC ... Done | MAE=0.1478 | AvgPredReturn=13.14%
[16/30] Training TNA

## 7. Profitability Scoring & Company Ranking

In [3]:
def compute_profitability_score(ticker, result, df_feat):
    """
    Composite profitability score combining:
    - avg_predicted_return : LSTM-predicted forward return (primary signal)
    - direction_accuracy   : how often the model predicts the correct direction
    - historical_sharpe    : annualized return / volatility on full history
    - recent_momentum      : 20-day price momentum (recent trend)
    """
    avg_ret = result['avg_predicted_return']
    dir_acc = result['direction_accuracy']

    daily_ret = df_feat['daily_return'].dropna()
    annual_ret = daily_ret.mean() * 252
    annual_vol = daily_ret.std() * np.sqrt(252)
    sharpe = annual_ret / (annual_vol + 1e-9)

    recent_close = df_feat['Close'].values
    if len(recent_close) >= 20:
        momentum = (recent_close[-1] - recent_close[-20]) / (recent_close[-20] + 1e-9)
    else:
        momentum = 0.0

    score = (
        0.40 * avg_ret +
        0.25 * (dir_acc - 0.5) +
        0.20 * sharpe / 10 +
        0.15 * momentum
    )

    return {
        'ticker': ticker,
        'avg_predicted_return': round(avg_ret * 100, 3),
        'direction_accuracy': round(dir_acc * 100, 2),
        'historical_sharpe': round(sharpe, 3),
        'recent_momentum_20d': round(momentum * 100, 3),
        'mae': round(result['mae'], 5),
        'rmse': round(result['rmse'], 5),
        'profitability_score': round(score, 6)
    }

score_rows = []
if 'all_results' not in globals():
    all_results = {}

for ticker, result in all_results.items():
    filepath = selected_meta.loc[selected_meta['ticker'] == ticker, 'filepath'].item()
    df = load_company_csv(filepath)
    df_feat = engineer_features(df)
    score_rows.append(compute_profitability_score(ticker, result, df_feat))

score_df = pd.DataFrame(score_rows).sort_values('profitability_score', ascending=False).reset_index(drop=True)
score_df.index += 1

print('=== Profitability Ranking (all trained companies) ===')
print(score_df.to_string())

NameError: name 'pd' is not defined

## 8. Portfolio Selection

In [ ]:
TOP_PORTFOLIO = 10

portfolio_df = score_df[
    (score_df['avg_predicted_return'] > 0) &
    (score_df['direction_accuracy'] >= 50)
].head(TOP_PORTFOLIO)

print(f'=== Selected Portfolio: Top {len(portfolio_df)} Profitable Companies ===')
print(portfolio_df[[
    'ticker', 'profitability_score',
    'avg_predicted_return', 'direction_accuracy',
    'historical_sharpe', 'recent_momentum_20d'
]].to_string())

portfolio_tickers = portfolio_df['ticker'].tolist()
print(f'\nPortfolio tickers: {portfolio_tickers}')

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2ecc71' if t in portfolio_tickers else '#95a5a6' for t in score_df['ticker']]
axes[0].barh(score_df['ticker'], score_df['profitability_score'], color=colors)
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Profitability Score')
axes[0].set_title('Profitability Score by Company\n(green = selected for portfolio)')
axes[0].invert_yaxis()

axes[1].scatter(
    score_df['historical_sharpe'],
    score_df['avg_predicted_return'],
    c=['#2ecc71' if t in portfolio_tickers else '#3498db' for t in score_df['ticker']],
    s=80, alpha=0.8
)
for _, r in score_df.iterrows():
    axes[1].annotate(r['ticker'], (r['historical_sharpe'], r['avg_predicted_return']),
                     fontsize=7, ha='left', va='bottom')
axes[1].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[1].axvline(0, color='red', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Historical Sharpe Ratio')
axes[1].set_ylabel('Avg Predicted 30-day Return (%)')
axes[1].set_title('Risk-Return Map\n(green = selected for portfolio)')

plt.tight_layout()
plt.savefig('task4_1_portfolio_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: task4_1_portfolio_overview.png')

In [ ]:
n_plot = min(6, len(portfolio_tickers))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, ticker in enumerate(portfolio_tickers[:n_plot]):
    result = all_results[ticker]
    y_true = result['y_true']
    y_pred = result['y_pred']
    dates = result['test_dates']
    n = min(len(y_true), len(y_pred), len(dates))

    ax = axes[idx]
    ax.plot(dates[:n], y_true[:n], label='Actual Return', color='#3498db', linewidth=1.5)
    ax.plot(dates[:n], y_pred[:n], label='Predicted Return', color='#e74c3c', linewidth=1.5, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.6, linestyle=':')
    ax.set_title(f'{ticker} — 30-day Forward Return')
    ax.set_ylabel('Return')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)

for j in range(n_plot, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top Portfolio Companies: Actual vs Predicted 30-day Returns (Test Set)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('task4_1_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: task4_1_predictions.png')

In [ ]:
ticker_sample = portfolio_tickers[0]
hist = all_results[ticker_sample]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(hist.history['loss'], label='Train Loss')
axes[0].plot(hist.history['val_loss'], label='Val Loss')
axes[0].set_title(f'{ticker_sample} — Training Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist.history['mae'], label='Train MAE')
axes[1].plot(hist.history['val_mae'], label='Val MAE')
axes[1].set_title(f'{ticker_sample} — Training MAE')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('task4_1_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: task4_1_training_curve.png')

## 10. Summary Report

In [ ]:
print('=' * 65)
print('TASK 4.1 — PROFITABLE STOCK SELECTION SUMMARY')
print('=' * 65)
print(f'Exchange       : {EXCHANGE_FILTER}')
print(f'Min data points: {MIN_DATA_POINTS}')
print(f'Companies loaded: {len(all_results)}')
print(f'Forecast horizon: {FORECAST_HORIZON} trading days (~6 weeks)')
print(f'Window size     : {WINDOW_SIZE} days')
print(f'Train/Val/Test  : {int(TRAIN_RATIO*100)}% / {int(VAL_RATIO*100)}% / {int((1-TRAIN_RATIO-VAL_RATIO)*100)}%')
print()
print('Model: Bidirectional LSTM (64) + GRU (32) + Dense')
print('Features: OHLCV + SMA10/20/50 + MACD + RSI14 + Volatility + Volume Ratio')
print('Target: 30-day forward return (regression)')
print()
print('Profitability Score Weights:')
print('  40% — LSTM predicted average future return')
print('  25% — Direction prediction accuracy')
print('  20% — Historical Sharpe ratio')
print('  15% — Recent 20-day momentum')
print()
print(f'SELECTED PORTFOLIO ({len(portfolio_df)} companies):')
print(portfolio_df[[
    'ticker', 'profitability_score',
    'avg_predicted_return', 'direction_accuracy', 'historical_sharpe'
]].to_string(index=True))
print()
print('Average portfolio metrics:')
print(f'  Avg predicted return : {portfolio_df["avg_predicted_return"].mean():.3f}%')
print(f'  Avg direction accuracy: {portfolio_df["direction_accuracy"].mean():.2f}%')
print(f'  Avg Sharpe ratio      : {portfolio_df["historical_sharpe"].mean():.3f}')
print('=' * 65)